# Section 3 — Agents That Remember (Local Ollama)

New local memory fundamentals demo. It uses two **fictional** user IDs, deterministic per-user dictionaries and a simple session reset. **Not** Mem0, LangGraph persistence or AWS AgentCore Memory. No real personal data or user authentication. Run Code Cells 1–8 sequentially.

## Cell 1 — Ollama preflight

In [ ]:
import json, urllib.request, time
from pathlib import Path

BASE = "http://127.0.0.1:11434"
MODEL = "qwen3:1.7b"
with urllib.request.urlopen(BASE + "/api/tags", timeout=10) as r:
    found = [x.get("name") for x in json.load(r).get("models", [])]
assert MODEL in found, "Run: ollama pull qwen3:1.7b"
print("PASS: Local Ollama available")

## Cell 2 — In-memory store for fictional users

In [ ]:
MEMORY = {}

def remember(user_id, key, value):
    if user_id not in {"sample_user_a", "sample_user_b"}:
        raise ValueError("Unknown fictional user")
    if not isinstance(key, str) or not isinstance(value, str):
        raise TypeError("Key and value must be text")
    MEMORY.setdefault(user_id, {})[key] = value
    return True

def recall(user_id):
    if user_id not in {"sample_user_a", "sample_user_b"}:
        raise ValueError("Unknown fictional user")
    return dict(MEMORY.get(user_id, {}))

print("PASS: In-memory per-user demo store ready")

## Cell 3 — Save fictional user preferences

In [ ]:
remember("sample_user_a", "favourite_city", "Jaipur")
remember("sample_user_a", "preferred_language", "Hindi")
remember("sample_user_b", "favourite_city", "Pune")
remember("sample_user_b", "preferred_language", "English")
assert len(MEMORY) == 2
print("PASS: Preferences stored for two invented users")

## Cell 4 — Verify isolation in demo API

In [ ]:
a_facts = recall("sample_user_a")
b_facts = recall("sample_user_b")
assert a_facts["favourite_city"] == "Jaipur"
assert b_facts["favourite_city"] == "Pune"
assert a_facts != b_facts
a_facts["favourite_city"] = "Changed local copy"
assert recall("sample_user_a")["favourite_city"] == "Jaipur"
print("PASS: Copy-based store separation (not production authorization)")

## Cell 5 — Build a context-bound question

In [ ]:
def context_for(user_id):
    memory = recall(user_id)
    # Server-side demo code selects user_id; do not accept it as a security credential.
    return "Fictional preferences:\n" + "\n".join(
        f"- {key}: {value}" for key, value in sorted(memory.items())
    )

context_a = context_for("sample_user_a")
context_b = context_for("sample_user_b")
assert "Jaipur" in context_a and "Pune" not in context_a
assert "Pune" in context_b and "Jaipur" not in context_b
print("PASS: Context is separated before sending to model")

## Cell 6 — Call Qwen3 with user-specific context

In [ ]:
def contextual_answer(context, question):
    payload = {
        "model": MODEL, "stream": False, "think": False,
        "messages": [
            {"role": "system", "content": "Use ONLY fictional facts supplied by the user context. Do not invent preferences."},
            {"role": "user", "content": context + "\n\nQuestion: " + question}
        ],
        "options": {"temperature": 0, "num_predict": 500},
    }
    for _ in range(2):
        req = urllib.request.Request(
            BASE + "/api/chat", data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json"}, method="POST"
        )
        with urllib.request.urlopen(req, timeout=210) as response:
            reply = str((json.load(response).get("message") or {}).get("content") or "").strip()
        if reply:
            return reply
        payload["options"]["num_predict"] = 700
    raise RuntimeError("Local Ollama returned empty content")

answer_a = contextual_answer(context_a, "Which city is my favourite?")
assert answer_a.strip()
print("Sample user A:", answer_a[:600])
print("PASS: Local context-bound answer produced")

## Cell 7 — Demonstrate session state and reset

In [ ]:
session_history = [{"role": "user", "content": "This is a fictional demo."}]
session_history.append({"role": "assistant", "content": answer_a})
assert len(session_history) == 2
session_history.clear()
assert session_history == []
assert len(MEMORY) == 2
print("PASS: Session cleared; separate long-term demo dictionary retained")

## Cell 8 — Save fictional memory test report

In [ ]:
report = {
    "section": 3, "model": MODEL,
    "scope": "In-memory per-user dictionary, short-term history, local context injection",
    "limits": "Not Mem0, LangGraph checkpointer, AgentCore Memory or production user authentication",
    "users_tested": 2,
    "user_a_context": context_a,
    "user_b_context": context_b,
    "user_a_model_answer": answer_a,
    "session_reset": (session_history == []),
}
path = Path("section3_local_memory_report.json")
path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
assert path.exists()
print("SAVED:", path.resolve())
print("PASS: Section 3 local memory demo")